# Deep Q-Networks
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/eleni-vasilaki/rl-notes/blob/main/notebooks/09_dqn.ipynb)


## To Table or Not to Table

Previously, we stored Q-values in a table. One entry per state-action pair. For our 6-state linear track with 2 actions, that's 12 numbers. For the cliff-walking grid (48 states, 4 actions) it's 192. Both are manageable. The agent can visit the relevant states many times, and the table can settle.

This is the great advantage of a table. Assuming the environment is Markovian, the agent explores enough, and the learning rate is chosen sensibly, each entry can move toward the correct value for that particular state-action pair. The table does not have to guess a shape for the Q-function. It does not matter whether it is smooth, jagged, or strange. Each entry is independent.

So if the state space is small enough, a table is hard to beat. A neural network with enough parameters could of course memorise a finite table, but then it is just doing the job of the table in a more complicated way. There is no gain unless the network can do something more useful: use structure in the problem to generalise.

The trouble begins when the state space grows. Consider a warehouse robot whose state includes its position on a 100 $\times$ 100 grid, its orientation, what it is carrying, and the locations of items around it. The number of possible states grows combinatorially; it can easily become billions of states. No table can hold that many entries in any practical sense, and even if it could, the agent would never visit most of them often enough.

So we need something else. But the replacement comes at a cost. A function approximator does not store every state-action pair independently. It assumes that there is some pattern to exploit. Whether that trade-off is worthwhile depends on whether the Q-function has structure.


## When to USE Artificial Neural Networks

Let us think what happened in the linear track. The Q-values for the "forward" action were not random. They increased smoothly with proximity to the goal. State 4 had a higher value than state 3, which was higher than state 2. There was a pattern. And because there was a pattern, a neural network could learn it: fit a function to a few states, and the function gives reasonable predictions for the rest. That's interpolation.

Now imagine a different situation. Suppose the reward at each state were assigned randomly -- high at state 2, low at state 3, high again at state 4, with no pattern whatsoever. The Q-function would look like noise. There would be nothing useful to interpolate, because the value at one state would carry no information about the value at any other state. A large enough network could memorise the examples it has seen, but it would not learn anything helpful about the ones it has not seen. At that point, a table is the honest solution.

This is a fundamental requirement: the Q-function must have enough regularity that knowing the value at some states tells you something about the value at nearby or similar states. If it does, a neural network can compress the Q-function into a small set of weights that generalise across the state space. If it does not, the network has nothing to generalise from.

The good news is that most real environments do have this structure. Physics is continuous. Nearby positions often lead to similar outcomes. Moving right from $(3, 5)$ in a grid world has roughly similar value to moving right from $(4, 5)$, because the two positions face similar geometry. The Q-function inherits some of the regularity of the environment.

This is what makes function approximation powerful: apart from using less memory than a table, it can learn about states it has barely visited, by exploiting the structure of the problem. The cost is that it can only do this when that structure exists.


## When to Use Deep Neural Networks

A single linear layer can only represent linear relationships. In the linear track, Q-values increase approximately linearly with proximity to the goal, so a simple model can already do something sensible. But many environments have structure that is not linear.

Consider a grid with walls. The optimal Q-values are not a simple function of position. They depend on which side of the wall the agent is on, whether a corridor is accessible, and how many turns are needed to reach the goal. The Q-function has **non-linear structure** that a single neuron cannot capture.

Adding hidden layers allows the network to compose simple features into complex ones. The first layer might learn to recognise which region of the grid the agent occupies. The second layer might combine region information with proximity to walls. The output layer then maps these composed features to Q-values. Each layer transforms the representation, making it progressively more suitable for the task.

This is what we mean by a **deep** network: multiple layers of learned transformations, where each layer builds on the previous one to capture increasingly abstract structure. The depth is what enables the network to represent the kinds of non-linear Q-functions that arise in complex environments.

However, in a simple problem such as a line track with few states, the complexity that comes with a neural network is not going to improve performance. Once you increase the number of states or you change the problem to navigation on a 2D space you may see advantages.


## Chain Rule for Gradients - Alternative Notation

We previously derived the backpropagation update rules element by element. We arrived at a weight update for any layer $k$:

$$
\Delta w_{jl}^{(k)} = \eta \, \delta_j^{(k)} \, x_l^{(k-1)},
$$

where the error signal $\delta$ was defined at the output layer as:

$$
\delta_i^{(n)} = (y_i^* - y_i) \, {f_i'}^{(n)},
$$

and at each hidden layer as:

$$
\delta_j^{(k)} = \left( \sum_m \delta_m^{(k+1)} \, w_{mj}^{(k+1)} \right) {f_j'}^{(k)}.
$$

This recursion works for any number of layers; the equations at every layer have the same form, and each layer's error is computed from the layer above. Effectively these calculations are only needed once for each input batch.

Now let us write the same equations in vector and matrix notation. Define $\boldsymbol{\delta}^{(k)}$ as the vector of all $\delta_j^{(k)}$, and $W^{(k)}$ as the weight matrix of layer $k$, where the element at row $j$, column $l$ is $w_{jl}^{(k)}$ -- the weight from neuron $l$ in layer $k-1$ to neuron $j$ in layer $k$.

Consider the sum $\sum_m \delta_m^{(k+1)} w_{mj}^{(k+1)}$ that appears in the recursion above. For a fixed $j$, this sums over all neurons $m$ in layer $k+1$, weighting each error $\delta_m^{(k+1)}$ by the connection $w_{mj}^{(k+1)}$ from neuron $j$ to neuron $m$. This is precisely the $j$-th element of the matrix-vector product ${W^{(k+1)}}^\top \boldsymbol{\delta}^{(k+1)}$. So the error recursion becomes:

$$
\boldsymbol{\delta}^{(k)} = \left( {W^{(k+1)}}^\top \boldsymbol{\delta}^{(k+1)} \right) \odot f'(\mathbf{h}^{(k)}),
$$

where $\odot$ denotes element-wise multiplication and $\mathbf{h}^{(k)}$ is the vector of pre-activations at layer $k$.

Similarly, the weight update $\Delta w_{jl}^{(k)} = \eta \, \delta_j^{(k)} \, x_l^{(k-1)}$ is the $(j, l)$ element of an outer product. Collecting all of them into a matrix:

$$
\Delta W^{(k)} = \eta \, \boldsymbol{\delta}^{(k)} \, {\mathbf{x}^{(k-1)}}^\top.
$$

These are the same equations we derived previously. The matrix form makes the computation efficient. Across all equations, the idea is the same: activity from the layer below times error from the layer above.

In the theory above, $W$ is written so that rows correspond to receiving neurons. In the NumPy code below, we store its transpose instead, because with a batch of inputs arranged row by row, the forward pass is then written naturally as $XW+b$. The link is the transpose identity: if for one column input vector the theory writes $h = Wx + b$, then for a batch of row vectors the same calculation can be written as $H = XW^\top + b$. For example, in the implementation below we write:

```python
h1 = X @ W1 + b1
```

which is exactly the same network, just stored with the transposed convention so that the batch calculation is easier to write. So transposing the stored matrix does not change the network; it only changes the convention used in the implementation.


## Using a Good Representation

How we represent the state to the network determines how easy generalisation is.

In the linear track, we used a one-hot encoding. State 3 in a 6-state track becomes:

$$[0, \; 0, \; 1, \; 0, \; 0,\; 0].$$

State 4 becomes:

$$[0, \; 0, \; 0, \; 1, \; 0, \; 0].$$

These two vectors share no activated components. They are orthogonal: as far as the input representation is concerned, they have nothing in common. In a single-layer network, when the agent updates its weights after visiting state 3, only the weights connected to position 3 in the input are affected. The output for state 4 does not change directly, because none of its input components were active.

With hidden layers, there can be some indirect sharing through later weights. But the important point remains: the representation itself has not told the network that state 3 and state 4 are neighbours. The network has to discover that from experience, which may be slow if experience is limited.

Now consider a different encoding. Instead of a single 1, we spread the activation across nearby positions. It becomes a bump centred on the current state, with smaller activations for its neighbours. State 3 becomes:

$$[0, \; 0.25, \; 1, \; 0.25, \; 0, \; 0],$$

and state 4 becomes:

$$[0, \; 0, \; 0.25, \; 1, \; 0.25, \; 0].$$

Now states 3 and 4 share activated components. When the network updates its weights after visiting state 3, the weights connected to positions 2, 3, and 4 are all adjusted. Because state 4's representation also activates positions 3 and 4, the output for state 4 is affected too. The network has generalised: learning about state 3 has informed it about state 4, without needing to treat the two states as completely separate.

This is the idea. A good representation creates **overlap** between similar states. That overlap gives the network continuity, i.e. the tendency to produce similar outputs for similar states. Without it, the network is pushed toward learning every state independently, much like a table.

In two dimensions, the same principle applies. A position $(r, c)$ on a grid can be encoded as a bump in a two-dimensional array, activating not just $(r, c)$ but also its spatial neighbours. Nearby grid positions then share activated components, and learning about one informs the others.

The choice of representation is not cosmetic. It determines whether the network can generalise from limited experience, or whether it has to spend most of its effort memorising the states it has already seen.


## Using the Right Parameters

### Weight Initialisation

Before training starts, a neural network has not learned anything. We still have to give every weight a starting value. This is what **weight initialisation** means.

It may seem natural to start with all weights equal to zero. This does not work well for hidden layers. If two hidden neurons start with exactly the same weights, they receive the same input, produce the same output, and receive the same update. They remain identical. So instead of having many useful hidden neurons, we have many copies of the same one.

For this reason, weights are usually initialised randomly. Randomness breaks the symmetry between neurons.

However, the random weights should not be too large or too small. If they are too large, the activity passing through the network may grow from layer to layer, and the updates can become unstable. If they are too small, the activity may shrink from layer to layer, and the early layers learn very slowly. So the goal is to choose random weights with a sensible scale.

The ReLU activation function is commonly used in neural networks and is defined as:

$$
f(h) = \max(0,h).
$$


Consider one neuron in a layer. It receives $n_{\text{in}}$ inputs, which we denote by $x_1, x_2, \ldots, x_{n_{\text{in}}}$. The corresponding weights are $w_1, w_2, \ldots, w_{n_{\text{in}}}$. Before the activation function, the neuron computes the weighted sum:

$$
h = \sum_{j=1}^{n_{\text{in}}} w_j x_j + b.
$$

The question is: how large should the random weights be?

We answer this by looking at the **variance** of the signal. We previously defined variance as a measure of how spread out the values of a random variable are around their mean. Here, the random variables are the values of the inputs and the values of the weights.

Let $X$ be the random variable describing the input values and $W$ the random variable describing the initial weight values. We can centre the inputs by preprocessing, so it is reasonable to take their mean to be 0. For the weights, we choose an initial distribution with mean 0. We can then write:

$$
E[X] = 0, \qquad \mathrm{Var}(X) = \sigma_x^2,
$$

and:

$$
E[W] = 0, \qquad \mathrm{Var}(W) = \sigma_w^2.
$$

So $\sigma_x^2$ is the variance of the input values, and $\sigma_w^2$ the variance of the weight values.

We now assume that each input $x_j$ is a sample from $X$, so after centring it has variance $\sigma_x^2$. That is,

$$
\mathrm{Var}(x_j) = \sigma_x^2,
$$

and that each initial weight $w_j$ is a sample from $W$, so it has variance $\sigma_w^2$. That is,

$$
\mathrm{Var}(w_j) = \sigma_w^2.
$$

We will also assume that the different inputs and weights are independent. This is the main approximation in the derivation.

For the derivation, we ignore the bias term because we initialise the biases to zero. So we focus on:

$$
h = \sum_{j=1}^{n_{\text{in}}} w_j x_j.
$$

We now calculate $\mathrm{Var}(h)$. Since $h$ is a sum, and we have already covered that the variance of a sum of independent variables is the sum of their variances, we obtain:

$$
\mathrm{Var}(h) = \sum_{j=1}^{n_{\text{in}}} \mathrm{Var}(w_j x_j).
$$

All these terms have the same variance, so:

$$
\mathrm{Var}(h) = n_{\text{in}} \, \mathrm{Var}(w x).
$$

So the problem reduces to calculating $\mathrm{Var}(w x)$.

With centred inputs and zero-mean initial weights, we have $E[w] = 0$ and $E[x] = 0$. Therefore:

$$
E[wx] = E[w]E[x] = 0,
$$

where we used the fact that for independent random variables the expectation of the product is the product of the expectations.

Now use the variance identity:

$$
\mathrm{Var}(w x) = E[(w x)^2] - (E[w x])^2.
$$

Since $E[w x] = 0$, this becomes:

$$
\mathrm{Var}(w x) = E[w^2 x^2].
$$

Again by independence:

$$
E[w^2 x^2] = E[w^2]E[x^2].
$$

And because both variables have mean 0, their variances are simply:

$$
\mathrm{Var}(w) = E[w^2], \qquad \mathrm{Var}(x) = E[x^2].
$$

Therefore:

$$
\mathrm{Var}(w x) = \sigma_w^2 \sigma_x^2.
$$

Substitute this back into the variance of $h$:

$$
\mathrm{Var}(h) = n_{\text{in}} \, \sigma_w^2 \sigma_x^2.
$$

This already tells us something important. If we keep the weight variance fixed and increase the number of inputs, the variance of the weighted sum grows in proportion to $n_{\text{in}}$. In plain words: the more inputs we add together, the easier it is for the signal to blow up.

If there were no activation function at all, and we simply wanted the variance of the signal to stay roughly the same from one layer to the next, we would ask for:

$$
\mathrm{Var}(h) \approx \sigma_x^2.
$$

Substituting the previous expression gives:

$$
n_{\text{in}} \, \sigma_w^2 \sigma_x^2 \approx \sigma_x^2,
$$

and therefore:

$$
\sigma_w^2 \approx \frac{1}{n_{\text{in}}}.
$$

But our hidden units are not linear. They use ReLU. ReLU sets every negative value of $h$ to zero. If the distribution of $h$ is roughly centred around zero, then about half the values are cut off. So after the ReLU, the activity is reduced. A rough but useful approximation is that ReLU keeps about half of the variance. In other words:

$$
\mathrm{Var}(f(h)) \approx \frac{1}{2} \mathrm{Var}(h).
$$

If we want the variance after ReLU to remain close to the variance of the inputs, we therefore ask for:

$$
\frac{1}{2} \mathrm{Var}(h) \approx \sigma_x^2.
$$

Substituting again $\mathrm{Var}(h) = n_{\text{in}} \, \sigma_w^2 \sigma_x^2$, we obtain:

$$
\frac{1}{2} n_{\text{in}} \, \sigma_w^2 \sigma_x^2 \approx \sigma_x^2.
$$

Cancel $\sigma_x^2$ on both sides:

$$
\frac{1}{2} n_{\text{in}} \, \sigma_w^2 \approx 1,
$$

which gives:

$$
\sigma_w^2 \approx \frac{2}{n_{\text{in}}}.
$$

This is the basic idea behind **He initialisation** (He, K. et al., "Delving Deep into Rectifiers: Surpassing Human-Level Performance on ImageNet Classification", *Proceedings of the IEEE International Conference on Computer Vision (ICCV)*, 2015). So, for a layer with $n_{\text{in}}$ inputs, we initialise each weight as:

$$
w \sim \mathcal{N}\left(0, \frac{2}{n_{\text{in}}}\right).
$$

For our DQN network, the first weight matrix connects the input representation to the hidden layer. If the input has size $n_{\text{in}}$ and the hidden layer has size $n_h$, then:

$$
W^{(1)} \in \mathbb{R}^{n_{\text{in}} \times n_h},
$$

and each element is drawn with variance $2/n_{\text{in}}$.

The second weight matrix connects the hidden layer to the output layer. If the output layer has size $n_{\text{out}}$, then:

$$
W^{(2)} \in \mathbb{R}^{n_h \times n_{\text{out}}},
$$

and now the relevant number of inputs is $n_h$, so each element is drawn with variance $2/n_h$.

The biases can be initialised to zero:

$$
b_j^{(1)} = 0, \qquad b_k^{(2)} = 0.
$$

This is acceptable because the random weights have already made the hidden neurons different from one another.

There are other initialisation methods. For sigmoid and tanh networks, a common reference is **Xavier initialisation** (Glorot, X. & Bengio, Y., "Understanding the difficulty of training deep feedforward neural networks", *Proceedings of the International Conference on Artificial Intelligence and Statistics (AISTATS)*, 2010). We do not need it for the exercise below, so we leave it as a reference only.

### Batch Normalisation

Xavier and He initialisation help at the start of training. But once training begins, the weights keep changing, and so the activations seen by deeper layers keep changing as well. A layer is then trying to learn while the distribution of its own inputs is drifting.

Instead of letting the activations wander freely in scale and mean, we normalise them within each mini-batch.

If $h_i$ denotes a pre-activation value in the current mini-batch, batch normalisation transforms it into:

$$
\hat{h}_i = \frac{h_i - \mu_B}{\sqrt{\sigma_B^2 + \epsilon}},
$$

where $\mu_B$ and $\sigma_B^2$ are the mean and variance computed over the mini-batch, and $\epsilon$ is a small constant added for numerical stability.

At first this may seem too restrictive. What if a layer would actually like its activations to have some other mean or scale? The standard way to give that freedom back is to introduce two learnable parameters, $\gamma$ and $\beta$:

$$
\tilde{h}_i = \gamma \hat{h}_i + \beta.
$$

So the point is not to force every layer to stay permanently at zero mean and unit variance. The point is to make optimisation easier by first stabilising the activations, while still allowing the network to learn the scale and shift that are useful.

How are $\gamma$ and $\beta$ learned? In the same way as the other parameters. Once the loss is defined, backpropagation gives gradients with respect to $\gamma$ and $\beta$ as well, and gradient descent updates them together with the weights and biases.

Batch normalisation is used a great deal in deep learning. In reinforcement learning it is also used, but usually with a little more care, because the data are not truly i.i.d. and the distribution changes during training.

This technique was introduced by Ioffe, S. & Szegedy, C. ("Batch Normalization: Accelerating Deep Network Training by Reducing Internal Covariate Shift", *Proceedings of the International Conference on Machine Learning (ICML)*, 2015).

### Normalisation of Inputs and Rewards

A simpler but equally important practice is normalising the inputs and, in reinforcement learning, keeping rewards on a sensible scale. If one input feature ranges from 0 to 1 and another from 0 to 10,000, the gradient with respect to the second feature will be thousands of times larger than the first. The optimiser will make huge steps in one direction and tiny steps in another. Normalising inputs to comparable scales puts the features on more equal footing.

In reinforcement learning, rewards can also vary enormously across environments. A game might give rewards of +1 for collecting a coin and -1000 for dying. The large negative reward can dominate the update. A very simple method is to clip rewards to $\{-1, 0, +1\}$. This throws away magnitude information, but it can make the same broad set of hyperparameters work across many environments.

### Reward Shaping

Sometimes the problem is not that the reward is too large or too small, but that it is too sparse. If the agent receives a useful reward only at the very end, learning may be painfully slow. In such cases, we often help the system by modifying the reward signal. This is called **reward shaping**.

Suppose the original reward is $r(s,a,s')$. We replace it by a shaped reward:

$$
\tilde{r}(s,a,s') = r(s,a,s') + F(s,s').
$$

The extra term $F(s,s')$ is there to guide learning.

For example, if we want the agent to find shorter solutions, we can give a small negative reward for each action. If we want to encourage progress, we can give intermediate rewards for reaching useful subgoals. If we want to discourage bad behaviour, we can penalise collisions, wasted moves, or getting stuck in unhelpful parts of the environment.

This can help any reinforcement-learning method, but it becomes especially relevant in harder problems, because those are exactly the cases where sparse rewards make learning too slow. Those harder problems are also the ones where we often need neural networks, so reward shaping is very relevant here.

But reward shaping must be done carefully. If we choose $F$ badly, we may accidentally change the problem itself, teaching the agent to optimise our shaping rewards rather than the task we actually care about.

A particularly important result is due to Ng, A. Y., Harada, D. & Russell, S. ("Policy Invariance Under Reward Transformations: Theory and Application to Reward Shaping", *Proceedings of the 16th International Conference on Machine Learning*, 1999). They showed that one safe and useful form is:

$$
F(s,s') = \gamma \Phi(s') - \Phi(s),
$$

$\Phi$ is a function defined on states. For each state $s$, it assigns one number, $\Phi(s)$. That number is not the true reward and not the true value function. It is an extra quantity we design in order to express a rough notion of how promising or how progressive a state is.

So $\Phi(s)$ can be higher in states that are closer to the goal, or in states that look more useful according to some heuristic. The same discount factor $\gamma$ appears here because we want this extra guidance to respect the same discounting from state to state as the original return. Then $F(s,s')$ rewards moves toward states with higher score and penalises moves toward states with lower score.

The important point is that this adds guidance without changing what the truly optimal policy is. So the agent may learn much faster, but in principle it is still being guided toward the same final solution.

## Overfitting, Underfitting, and Over-parameterisation

Before going further, it is worth pausing on a general difficulty with neural networks. What a neural network is trying to do, at bottom, is to learn a function from samples.

If the samples cover the input space well enough, the network may learn a function that interpolates sensibly between them. If they do not, then many different functions may agree with the samples we have already seen, and the network has too little information to decide which one is the right one.

This is where **underfitting** and **overfitting** appear.

If the model is too simple, it **underfits**. It cannot capture the structure of the function even in the regions where we do have data. A straight line cannot fit a curved value function. A tiny network may not represent the difference between being on one side of a wall and being on the other.

If the model has too many free parameters relative to the amount of data we have, and the data do not constrain it well enough, it may **overfit**. Then the network does not really learn the underlying function; it learns the particular samples, including their noise or accidents. It looks very good on the data it has already seen, but performs badly on new data.

How do we detect this in ordinary supervised learning? We keep aside a **test set**: data that are statistically independent from the training set and have never been used to fit the network. The test set should still come from the same kind of problem, otherwise the comparison is unfair. We often also keep a **validation set** for tuning hyperparameters, while the test set is left untouched until the very end.

### Over-parameterisation

This classical picture suggests a simple intuition: larger networks should overfit more easily. But modern deep learning complicates this story. Very large networks can sometimes generalise well even when they have enough capacity to fit the training data exactly.

One possible intuition is geometric. Hidden layers map the data into new spaces. In high-dimensional spaces, points can become easier to separate, and the network may fit the training data without needing a highly oscillatory function. This geometric idea goes back at least to Cover, T. M., "Geometrical and Statistical Properties of Systems of Linear Inequalities with Applications in Pattern Recognition", *IEEE Transactions on Electronic Computers*, 14(3), 326-334, 1965.

Related modern ideas here include **benign overfitting** (Bartlett, P. L. et al., "Benign Overfitting in Linear Regression", *Proceedings of the National Academy of Sciences*, 117(48), 30063-30070, 2020) and the **double descent** curve, where test error can first increase and then decrease again as the model becomes more over-parameterised (Belkin, M. et al., "Reconciling modern machine learning practice and the bias-variance trade-off", *Proceedings of the National Academy of Sciences*, 116(32), 15849-15854, 2019).

This does not mean that size no longer matters, or that any large network will generalise well. It means that the old intuition -- more parameters automatically implies worse generalisation -- is no longer reliable on its own. Optimisation, architecture, data structure, and implicit regularisation all matter.

### What Changes in Reinforcement Learning?

In reinforcement learning, the story is a little different because there is no fixed training set waiting for us. The agent creates its own samples by acting, and as the policy changes, the states it visits also change. So the question is not whether overfitting exists, but how it appears in this setting.

Still, the basic capacity question does not disappear. A model may be too simple to capture the structure of the Q-function. Or it may be so large that it requires more data to be constrained properly. In reinforcement learning, that matters because the agent has to generate those data through experience, so learning may become slower, more data-hungry, or simply impractical. So the size of the network still matters.

There is also an important difference in what matters. In supervised learning, we usually want a function that generalises across the whole input distribution. In reinforcement learning, errors in states the agent will never visit may not matter very much. What matters is whether the learned values are good enough in the parts of the state-action space that actually control behaviour.

So in reinforcement learning, overfitting is not absent; it just changes form. A network may work well only for the trajectories, random seeds, or environment settings it happened to see during training. This is one reason why, when possible, it is useful to evaluate on independent runs, different seeds, or modified environments. In fact, an algorithm should not be judged from one seed, and fixing a single seed is in general a bad idea for evaluation, even though it may be encouraged for reproducibility. Reproducibility in reinforcement learning should be statistical rather than exact: the correct process is to repeat the experiment many times and confirm that it works.

A larger network is not automatically worse, and a smaller network is not automatically better. The real question is whether the network has the right capacity for the Q-function we are trying to learn, and whether the agent can collect enough representative experience to support that learning.


# Exercise

The following code fits polynomial models of increasing degree to a noisy sine wave. This is not a reinforcement-learning example; it is just a small supervised-learning picture of underfitting and overfitting. Keep the analogy, but do not take it too literally. In reinforcement learning, the data distribution moves because the agent changes its behaviour.

Run the code and observe how the training and validation errors change as the model becomes more complex. The test set is left aside until the end and used only as a final check.

- At which degree is the validation error smallest?
- What happens when the number of parameters approaches the number of training points?
- Try increasing `n_train` to 50 or 100. How does this affect the onset of overfitting?
- Try increasing `noise_level` to 0.5 or 1.0. Does the validation "sweet spot" degree change?


In [ ]:
import numpy as np
import matplotlib.pyplot as plt


def polynomial_fit_demo(n_train=15, n_val=60, n_test=200, noise_level=0.3, max_degree=14, seed=42):
    """
    Fit polynomials of increasing degree to noisy sine data
    and plot training vs validation error.
    A held-out test set is used only at the end.

    Args:
        n_train (int): Number of training points.
        n_val (int): Number of validation points.
        n_test (int): Number of test points.
        noise_level (float): Standard deviation of Gaussian noise.
        max_degree (int): Maximum polynomial degree to try.
        seed (int): Random seed for reproducibility.
    """
    rng = np.random.default_rng(seed)

    # Generate data
    x_train = np.sort(rng.uniform(0, 2 * np.pi, n_train))
    y_train = np.sin(x_train) + rng.normal(0, noise_level, n_train)
    x_val = np.sort(rng.uniform(0, 2 * np.pi, n_val))
    y_val = np.sin(x_val) + rng.normal(0, noise_level, n_val)
    x_test = np.sort(rng.uniform(0, 2 * np.pi, n_test))
    y_test = np.sin(x_test) + rng.normal(0, noise_level, n_test)
    x_plot = np.linspace(0, 2 * np.pi, 300)
    y_plot = np.sin(x_plot)

    # Normalise x to [-1, 1] for numerical stability
    x_mid = (x_train.max() + x_train.min()) / 2
    x_half = (x_train.max() - x_train.min()) / 2
    xn_train = (x_train - x_mid) / x_half
    xn_val = (x_val - x_mid) / x_half
    xn_test = (x_test - x_mid) / x_half
    xn_plot = (x_plot - x_mid) / x_half

    def poly_features(x, degree):
        return np.column_stack([x**k for k in range(degree + 1)])

    degrees = list(range(1, min(max_degree + 1, n_train)))
    train_errors, val_errors = [], []
    models = {}

    for deg in degrees:
        P_tr = poly_features(xn_train, deg)
        P_val = poly_features(xn_val, deg)
        w, _, _, _ = np.linalg.lstsq(P_tr, y_train, rcond=None)

        models[deg] = w
        train_errors.append(np.mean((P_tr @ w - y_train)**2))
        val_errors.append(np.mean((P_val @ w - y_val)**2))

    best_idx = int(np.argmin(val_errors))
    best_degree = degrees[best_idx]
    w_best = models[best_degree]
    P_test = poly_features(xn_test, best_degree)
    test_error = np.mean((P_test @ w_best - y_test)**2)

    # Plot 1: train vs validation error
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    axes[0].semilogy(degrees, train_errors, 'o-', label='Training error')
    axes[0].semilogy(degrees, val_errors, 's-', label='Validation error')
    axes[0].axvline(x=n_train - 1, color='gray', linestyle=':', label=f'n_params = n_train ({n_train})')
    axes[0].axvline(x=best_degree, color='tab:green', linestyle='--', alpha=0.8, label=f'Best degree ({best_degree})')
    axes[0].set_xlabel('Polynomial degree')
    axes[0].set_ylabel('Mean Squared Error (log scale)')
    axes[0].set_title('Overfitting: Training vs Validation Error')
    axes[0].legend()
    axes[0].grid(True)
    axes[0].text(0.03, 0.05,
                 f'Held-out test MSE at degree {best_degree}: {test_error:.3f}',
                 transform=axes[0].transAxes,
                 fontsize=9,
                 bbox=dict(facecolor='white', alpha=0.8, edgecolor='lightgray'))

    # Plot 2: fitted curves at key degrees
    key_degrees = []
    for deg in [1, best_degree, max(degrees)]:
        if deg not in key_degrees:
            key_degrees.append(deg)

    axes[1].scatter(x_train, y_train, color='black', s=30, zorder=5, label='Training data')
    axes[1].plot(x_plot, y_plot, 'k--', alpha=0.3, label='True function')

    for deg in key_degrees:
        P_tr = poly_features(xn_train, deg)
        P_plot = poly_features(xn_plot, deg)
        w, _, _, _ = np.linalg.lstsq(P_tr, y_train, rcond=None)
        y_pred = P_plot @ w
        if deg == 1:
            label = 'Underfitting'
            color = 'blue'
        elif deg == best_degree:
            label = 'Best validation fit'
            color = 'green'
        else:
            label = 'Overfitting'
            color = 'red'
        axes[1].plot(x_plot, np.clip(y_pred, -3, 3), color=color, label=f'Degree {deg} ({label})')

    axes[1].set_xlabel('x')
    axes[1].set_ylabel('y')
    axes[1].set_ylim(-3, 3)
    axes[1].set_title('Fitted Polynomials')
    axes[1].legend()
    axes[1].grid(True)

    plt.tight_layout()
    plt.show()


# Run the demo — try changing the parameters!
polynomial_fit_demo(n_train=15, noise_level=0.3, max_degree=14)


## Adam: Adaptive Learning Rates

Throughout this course, we have used a single learning rate $\eta$ for all parameters. The basic gradient update is:

$$
\theta_t = \theta_{t-1} - \eta \, g_t.
$$

This already shows why the learning rate matters so much. If $\eta$ is small, learning is slow. If $\eta$ is large, the update may overshoot a local minimum or make the parameters oscillate around it.

Over the years, many techniques were developed to reduce this sensitivity. Some smooth the gradient over time, so the update follows a more reliable direction. Others rescale the gradient, so that parameters with large gradients take smaller steps while parameters with small gradients do not get stuck. In that sense, these methods adapt the effective learning rate during training instead of keeping it fixed once and for all.

Adam combines these ideas and, for that reason, became a standard optimiser.

The first term it keeps is a running average of the gradient itself:

$$
m_t = \beta_1 \, m_{t-1} + (1 - \beta_1) \, g_t.
$$

This is an exponentially decaying running average. The new gradient $g_t$ is included, but the past is not forgotten immediately. Instead, older gradients are kept with coefficients that decay geometrically over time. So $m_t$ is a smoothed version of the gradient. This is why it behaves a bit like momentum.

The parameter $\beta_1$ controls how much memory this running average has. If $\beta_1$ is close to 1, the average changes slowly and remembers the past for longer. If it is smaller, the average reacts more quickly to the current gradient. The usual default is $\beta_1 = 0.9$.

The second term Adam keeps is a running average of the squared gradient:

$$
v_t = \beta_2 \, v_{t-1} + (1 - \beta_2) \, g_t^2.
$$

This is again an exponentially decaying running average, but now of the squared gradient. Here we are interested in the magnitude of the gradient. If we averaged the raw gradient again, positive and negative values could cancel. By squaring it, large gradients contribute strongly regardless of sign. So $v_t$ tracks how large the gradients have been recently. If a parameter has been receiving large gradients, $v_t$ becomes large. If it has been receiving small gradients, $v_t$ stays small.

The parameter $\beta_2$ plays the same role here: it sets the time scale of the running average. The usual default is $\beta_2 = 0.999$, which means this average changes more slowly and keeps a longer memory than $m_t$.

Now we come to the bias correction. If you look at the recurrence, every new gradient enters $m_t$ multiplied by $(1 - \beta_1)$; the running average is systematically scaled down. This is a problem particularly at the beginning, since $m_0$ is initialised at zero.

For the same reason, $v_t$ is also initially too small. Adam corrects this by dividing both running averages by factors that scale them back up:

$$
\hat{m}_t = \frac{m_t}{1 - \beta_1^t}, \qquad \hat{v}_t = \frac{v_t}{1 - \beta_2^t}.
$$

If $g_t$ is the gradient of the loss, the update is:

$$
\theta_t = \theta_{t-1} - \eta \, \frac{\hat{m}_t}{\sqrt{\hat{v}_t} + \epsilon},
$$

where $\epsilon \approx 10^{-8}$ prevents division by zero. So the numerator gives a smoothed update direction, while the denominator normalises that direction by the recent size of the gradient.

Adam was introduced by Kingma, D. P. & Ba, J. ("Adam: A Method for Stochastic Optimization", *Proceedings of the International Conference on Learning Representations (ICLR)*, 2015). It is now a default optimiser in many deep learning applications, reinforcement learning included. We do not implement it in this notebook -- our exercises use fixed-rate SGD.


## When Things Become Complex

There are two problems that arise when combining neural networks with Q-learning. 

### Catastrophic Forgetting and the Replay Buffer

When training a neural network with gradient descent, the training data must cover the distribution the network needs to learn. If you train a digit recogniser by showing it only threes, then only sevens, then only fives, it will keep overwriting what it learned about previous digits. Shuffle the data and it behaves much better. This is a general property of gradient-based learning, not something special to reinforcement learning.

In reinforcement learning, the agent generates its own training data by acting in the environment. As it plays, the transitions it collects are not representative of all possible experience. They are biased toward the agent's current trajectory. If the agent is near state 3, the next several transitions will all involve states around 3 and 4. Training immediately on these transitions means the network is repeatedly pushed to fit one region of the state space, possibly at the expense of others.

This phenomenon where learning new information degrades previously learned knowledge is called **catastrophic forgetting**. It is not specific to reinforcement learning; it affects any neural network trained sequentially on data whose distribution keeps changing.

**Solution: Experience Replay.** Instead of training on each transition as it arrives, the agent stores transitions in a circular buffer:

$$
\mathcal{D} = \{(s, a, r, s', \text{done})_1, (s, a, r, s', \text{done})_2, \ldots\}.
$$

At each training step, it samples a random mini-batch from this buffer. The buffer contains transitions from many different episodes and regions of the state space, so a random sample from it is much less correlated than the last few consecutive transitions. It is also more representative of the broader distribution the network needs to learn. This is very similar to shuffling training data in supervised learning. It is not perfect, because the buffer still reflects what the agent has experienced, but it is far better than learning only from the most recent steps.

Replay buffers are sometimes compared to **hippocampal replay** in neuroscience: the process by which the brain replays past experiences during rest, believed to help consolidate memories. The computational motivation is similar: revisit past experience so that new learning does not immediately wipe out old learning.

### The Moving Target Problem

The TD target used in Q-learning is:

$$
y_a^*(s) = r + \gamma \max_{a'} Q(s', a'; W).
$$

The problem is that the target depends on the same network we are updating. We change $W$ to make the network closer to the target, but changing $W$ also changes the target. The network is trying to hit something that moves every time the network takes a step.

This kind of problem appears in many systems where two things adapt to each other. A changes B, but B also changes with A. If both move too quickly, they can chase each other, oscillate, or diverge. One common way to stabilise such systems is to separate their time scales: let one part move quickly and the other move slowly.

DQN uses this separation of time scales in a very direct way.

**Solution: Target Network.** Maintain a separate copy of the network weights, $W^{-}$, used only for computing targets:

$$
y_a^*(s) = r + \gamma \max_{a'} Q(s', a'; W^{-}).
$$

The current network $W$ is updated at every training step. The target network $W^{-}$ is frozen for many steps. During that time, the target is much more stable, because it is computed from weights that are not changing. Then, every $C$ steps, we copy the current network into the target network:

$$
W^{-} \leftarrow W.
$$

So the basic idea is to separate the two time scales. One network learns quickly; the other stands still for a while. Then the still network is refreshed, and the process repeats.

A softer version of the same idea is to update the target network slowly:

$$
W^{-} \leftarrow \tau W + (1 - \tau) W^{-}, \qquad \tau \ll 1.
$$

But in the original DQN, the approach is simple and effective: freeze the target network, learn against it, then copy.


## The DQN Algorithm

Combining the neural-network Q-learning with experience replay and a target network gives us the **Deep Q-Network (DQN)** algorithm (Mnih, V. et al., "Human-level control through deep reinforcement learning", *Nature*, 518, 529-533, 2015).

The algorithm proceeds as follows.

Initialise the Q-network with weights $W$ and the target network with $W^{-} = W$. Initialise the replay buffer $\mathcal{D}$.

For each episode:
  - Observe state $s$.
  - Select action $a$ using epsilon-greedy: with probability $\epsilon$ choose a random action; otherwise choose $a = \arg\max_{a'} Q(s, a'; W)$.
  - Execute $a$, observe reward $r$, next state $s'$, and whether the episode is done.
  - Store $(s, a, r, s', \text{done})$ in $\mathcal{D}$.
  - Sample a random mini-batch of transitions from $\mathcal{D}$.
  - For each transition in the batch, compute the target for the action that was actually taken: $y_a^* = r$ if done, otherwise $y_a^* = r + \gamma \max_{a'} Q(s', a'; W^{-})$.
  - Leave the targets for the other actions equal to their current Q-values. This means the update changes the selected action and leaves the non-selected actions alone.
  - Update $W$ by performing a gradient step to reduce the squared error between the network output and this target array.
  - Every $C$ steps, synchronise: $W^{-} \leftarrow W$.
  - Decay $\epsilon$.

The target number is treated as fixed during the update. We compute it, then we ask the current network to move toward it. We do not try to update the target network through this gradient step; the target network only changes when we explicitly copy $W$ into $W^{-}$.

This is the standard DQN algorithm. The epsilon-greedy action-selection line above, $a = \arg\max_{a'} Q(s, a'; W)$, is used in both DQN and Double DQN. The difference does not appear there. It appears only in the target line for $s'$. Double DQN keeps the same overall structure, but changes that target so that the current network chooses the next action and the target network evaluates it.

The gradient update in the batch step is exactly the backpropagation we implemented in notebook 08, applied over a mini-batch rather than a single transition.


## Examples of Deep Q-Learning

### Historical Examples

DQN's landmark achievement was learning to play Atari 2600 games from raw pixel inputs (Mnih, V. et al., "Human-level control through deep reinforcement learning", *Nature*, 518, 529-533, 2015). The network received a stack of four consecutive 84 $\times$ 84 grayscale frames as input -- stacking captures motion, which a single frame cannot -- and produced Q-values for each joystick action as output.

A separate network was trained for each game, but the architecture, learning algorithm, and hyperparameters were kept the same across the 49 games. That was the remarkable point: not one hand-designed controller per game, but the same learning machinery applied again and again. On many games, DQN reached or exceeded human-level performance.

A key follow-up was **Double DQN** (van Hasselt, H., Guez, A. & Silver, D., "Deep Reinforcement Learning with Double Q-learning", *Proceedings of the AAAI Conference on Artificial Intelligence*, 2016). This can be confusing at first, because standard DQN already has two networks: the current network $W$ and the target network $W^{-}$. So Double DQN is not about adding another network. Standard DQN uses the second network to stabilise learning. Double DQN changes what those two networks do inside the target. In standard DQN, the target $\max_{a'} Q(s', a'; W^{-})$ uses the target-network values both to choose the action and to evaluate it. If those values are noisy, the max tends to favour an action whose value has been overestimated.

Double DQN separates those two roles. The current network chooses the action, and the target network evaluates the action that was chosen:

$$
y_a^* = r + \gamma \, Q\!\left(s',\; \arg\max_{a'} Q(s', a'; W);\; W^{-}\right).
$$

So the two-network structure was already present in DQN. Double DQN uses that structure more carefully, and this substantially reduces overestimation.


### Contemporary Examples

DQN was introduced in 2015; however, it is still used in contemporary applications. This is especially true when the action space is discrete and one wants a method that is relatively simple, stable, and well understood.

**Autonomous robotic ultrasound.** Su, K. et al. ("A fully autonomous robotic ultrasound system for thyroid scanning", *Nature Communications*, 15, 4004, 2024) describe a medical robotics system for autonomous thyroid ultrasound scanning. One part of the problem is a search task: from the current ultrasound image, the robot must decide how to move the probe so that it can find the thyroid before carrying out the full scan. DQN is used for that sequential decision problem. This is a good example of how DQN is often used in practice: not as the whole system, but as one decision-making component inside a larger pipeline.

**Clinical bias mitigation.** Yang, J. et al. ("Algorithmic fairness and bias mitigation for clinical machine learning with deep reinforcement learning", *Nature Machine Intelligence*, 5, 884-894, 2023) use a duelling Double DQN in a clinical prediction setting. Here the task is not robot motion, but sequential decision-making under a reward that balances predictive accuracy with fairness across hospitals and ethnic groups. Their main application was rapid COVID-19 screening in emergency departments, with additional experiments on ICU discharge prediction. Again, the attraction is clear: the decisions are discrete, the objective is custom-designed, and the value function can still be learned with a neural network.

In both cases, the basic idea is still the one developed in this notebook: estimate action values with a neural network, improve them through experience, and stabilise training with a target network or a close variant of it.


# Exercise

Implement a `ReplayBuffer` class that stores transitions and supports random sampling. This data structure is the foundation of DQN — it decouples data collection from training, enabling the representative mini-batches that gradient descent requires.

The buffer should have a fixed maximum capacity. When full, new transitions overwrite the oldest ones (circular buffer). The `sample` method should return a random mini-batch of transitions.

Think about what the `store` and `sample` methods need to do, and what data structure is appropriate for constant-time insertion and random access.


In [ ]:
import numpy as np


class ReplayBuffer:
    """Fixed-capacity circular buffer for storing transitions."""

    def __init__(self, capacity):
        """
        Args:
            capacity (int): Maximum number of transitions to store.
        """
        # TODO: initialise the buffer, position counter, and capacity
        pass

    def store(self, state, action, reward, next_state, done):
        """
        Store a transition. If the buffer is full, overwrite the oldest entry.

        Args:
            state (int): Current state.
            action (int): Action taken.
            reward (float): Reward received.
            next_state (int): Next state.
            done (bool): Whether the episode ended.
        """
        # TODO: implement circular buffer storage
        pass

    def sample(self, batch_size):
        """
        Sample a random mini-batch of transitions.

        Args:
            batch_size (int): Number of transitions to sample.

        Returns:
            Tuple of arrays: (states, actions, rewards, next_states, dones)
        """
        # TODO: sample random indices and return arrays
        pass

    def __len__(self):
        """Return the current number of stored transitions."""
        # TODO
        pass


# === Test your implementation ===
# buf = ReplayBuffer(capacity=100)
# for i in range(150):
#     buf.store(state=i % 10, action=i % 4, reward=float(i), next_state=(i+1) % 10, done=(i % 10 == 9))
# print(f"Buffer size: {len(buf)} (should be 100)")
# states, actions, rewards, next_states, dones = buf.sample(8)
# print(f"Batch states: {states}")
# print(f"Batch actions: {actions}")


<details>
<summary>Show Solution</summary>

```python
import numpy as np


class ReplayBuffer:
    """Fixed-capacity circular buffer for storing transitions."""

    def __init__(self, capacity):
        self.capacity = capacity
        self.buffer = []
        self.position = 0

    def store(self, state, action, reward, next_state, done):
        transition = (state, action, reward, next_state, done)
        if len(self.buffer) < self.capacity:
            self.buffer.append(transition)
        else:
            self.buffer[self.position] = transition
        self.position = (self.position + 1) % self.capacity

    def sample(self, batch_size):
        indices = np.random.choice(len(self.buffer), batch_size, replace=False)
        batch = [self.buffer[i] for i in indices]
        states = np.array([t[0] for t in batch])
        actions = np.array([t[1] for t in batch])
        rewards = np.array([t[2] for t in batch])
        next_states = np.array([t[3] for t in batch])
        dones = np.array([t[4] for t in batch], dtype=float)
        return states, actions, rewards, next_states, dones

    def __len__(self):
        return len(self.buffer)


# === Test ===
buf = ReplayBuffer(capacity=100)
for i in range(150):
    buf.store(state=i % 10, action=i % 4, reward=float(i), next_state=(i+1) % 10, done=(i % 10 == 9))
print(f"Buffer size: {len(buf)} (should be 100)")
states, actions, rewards, next_states, dones = buf.sample(8)
print(f"Batch states: {states}")
print(f"Batch actions: {actions}")
```

</details>


# Exercise

Implement a full DQN agent on a grid-world environment. The environment, a `ReplayBuffer` (from the previous exercise), and a `DQNAgent` skeleton are provided.

Complete the following four methods:

- `_sync_target_network`: copy the current network weights ($W_1$, $b_1$, $W_2$, $b_2$) into separate target network copies ($W_1^{-}$, $b_1^{-}$, $W_2^{-}$, $b_2^{-}$). This is called every `target_update_freq` steps.
- `_forward`: forward pass through a two-layer network with ReLU hidden units and **linear output** (no activation on the final layer — Q-values can be positive or negative). The `use_target` flag selects which set of weights to use. Return both the Q-values and a cache of intermediates needed for backpropagation.
- `_backward`: given the cache from `_forward` and a target Q-value array, compute the output error, backpropagate through the ReLU layer, and update $W_1$, $b_1$, $W_2$, $b_2$. This follows the same pattern as notebook 08's `TwoLayerQAgent._backward`, but applied to a mini-batch.
- `update`: sample a mini-batch from the replay buffer. Compute Q-values for current states using the current network, and Q-values for next states using the target network. Build the target array by setting $y_a^*(s) = r$ if done, or $y_a^*(s) = r + \gamma \max_{a'} Q(s', a'; W^{-})$ otherwise, leaving the other actions' targets equal to the current Q-values. Then call `_backward`.

The environment is a 5 $\times$ 5 grid with a vertical wall and a single gap. The agent starts at the top-left corner and must reach the bottom-right corner. It receives a reward of $+10$ upon reaching the goal and $-1$ for each step.

**A note on didactic honesty:** This grid world is small enough (25 states, 4 actions) that tabular Q-learning from notebook 06 would solve it efficiently. DQN is not necessary here, and the replay buffer and target network will not provide a dramatic visible advantage over the simpler neural-network Q-learning from notebook 08. We use this environment because it is small enough to train quickly in NumPy, while still requiring the agent to navigate around an obstacle. The purpose of this exercise is to practise the implementation of the DQN components, which become essential in larger environments where simpler methods fail.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt


def relu(x):
    """ReLU activation function."""
    return np.maximum(0, x)


def relu_derivative(x):
    """Derivative of ReLU, evaluated at the pre-activation x."""
    return (x > 0).astype(float)


class GridWorldEnvironment:
    """5x5 grid world with a vertical wall and a gap.

    The agent starts at (0, 0) and must reach (4, 4).
    A wall runs down column 2 with a gap at row 2.
    """

    def __init__(self):
        self.grid_size = 5
        self.num_states = 25
        self.actions = 4  # 0=up, 1=down, 2=left, 3=right
        self.start = (0, 0)
        self.goal = (4, 4)
        self.moves = {0: (-1, 0), 1: (1, 0), 2: (0, -1), 3: (0, 1)}

        # Vertical wall at column 2, gap at row 2
        self.walls = set()
        for r in range(self.grid_size):
            if r != 2:
                self.walls.add((r, 2))

    def reset(self):
        """Reset to start position. Returns state index."""
        self.state = self.start
        return self.state[0] * self.grid_size + self.state[1]

    def step(self, action):
        """Take an action. Returns (state_index, reward, done)."""
        dr, dc = self.moves[action]
        next_pos = (self.state[0] + dr, self.state[1] + dc)

        if (0 <= next_pos[0] < self.grid_size and
            0 <= next_pos[1] < self.grid_size and
            next_pos not in self.walls):
            self.state = next_pos

        idx = self.state[0] * self.grid_size + self.state[1]

        if self.state == self.goal:
            return idx, 10.0, True
        return idx, -1.0, False


class DQNAgent:
    """DQN agent with experience replay and target network."""

    def __init__(self, env, hidden_size=32, eta=0.01, gamma=0.95,
                 epsilon=1.0, epsilon_min=0.01, epsilon_decay=0.99,
                 buffer_size=3000, batch_size=32, target_update_freq=200):
        self.env = env
        self.hidden_size = hidden_size
        self.eta = eta
        self.gamma = gamma
        self.epsilon = epsilon
        self.epsilon_min = epsilon_min
        self.epsilon_decay = epsilon_decay
        self.batch_size = batch_size
        self.target_update_freq = target_update_freq
        self.buffer = ReplayBuffer(buffer_size)
        self._reset_weights()

    def _reset_weights(self):
        """Initialise network weights using He initialisation."""
        n_in = self.env.num_states
        n_h = self.hidden_size
        n_out = self.env.actions

        self.W1 = np.random.randn(n_in, n_h) * np.sqrt(2.0 / n_in)
        self.b1 = np.zeros(n_h)
        self.W2 = np.random.randn(n_h, n_out) * np.sqrt(2.0 / n_h)
        self.b2 = np.zeros(n_out)

        self._sync_target_network()

    def _sync_target_network(self):
        """
        Copy current network weights to the target network.

        TODO: create copies of W1, b1, W2, b2 as W1_target, b1_target, etc.
        """
        pass

    def _one_hot_batch(self, states):
        """Encode a batch of state indices as one-hot vectors."""
        X = np.zeros((len(states), self.env.num_states))
        X[np.arange(len(states)), states] = 1
        return X

    def _forward(self, X, use_target=False):
        """
        Forward pass through the two-layer network.

        Args:
            X (ndarray): One-hot encoded states, shape (batch, num_states).
            use_target (bool): If True, use target network weights.

        Returns:
            (q_values, cache): Q-values and dict with intermediates for backprop.

        TODO: select weights based on use_target flag.
        Compute h1 = X @ W1 + b1, hidden = relu(h1), q = hidden @ W2 + b2.
        Return (q, {'X': X, 'h1': h1, 'hidden': hidden}).
        """
        pass

    def _backward(self, cache, targets):
        """
        Backward pass: update current network weights.

        Args:
            cache (dict): Intermediates from _forward (X, h1, hidden).
            targets (ndarray): Target Q-values, shape (batch, actions).

        TODO: recompute q from cache, compute delta_output = targets - q,
        backpropagate through ReLU, update W2, b2, W1, b1.
        """
        pass

    # The epsilon-greedy logic lives directly in this method.
    # Notebook 04 shows an alternative: passing a separate policy
    # object into the agent, so you can swap policies without
    # changing the agent class.
    def select_action(self, state):
        """Epsilon-greedy action selection."""
        if np.random.rand() < self.epsilon:
            return np.random.randint(self.env.actions)
        X = self._one_hot_batch(np.array([state]))
        q, _ = self._forward(X)
        return np.argmax(q[0])

    def update(self):
        """
        Sample a mini-batch from the replay buffer and update the network.

        TODO:
        1) Return early if buffer has fewer transitions than batch_size.
        2) Sample a batch from the buffer.
        3) Compute Q-values for current states (current network).
        4) Compute Q-values for next states (target network).
        5) Build targets: copy current Q-values, then for each transition
           set targets[i, action] = reward if done,
           else reward + gamma * max(next_q[i]).
        6) Call _backward with cache and targets.
        """
        pass

    def train(self, episodes=500, max_steps=100):
        """Train the agent. Returns list of steps per episode."""
        steps_per_episode = []
        total_steps = 0

        for episode in range(episodes):
            state = self.env.reset()
            done = False
            steps = 0

            while not done and steps < max_steps:
                action = self.select_action(state)
                next_state, reward, done = self.env.step(action)
                self.buffer.store(state, action, reward, next_state, done)
                self.update()
                state = next_state
                steps += 1
                total_steps += 1

                if total_steps % self.target_update_freq == 0:
                    self._sync_target_network()

            self.epsilon = max(self.epsilon_min, self.epsilon * self.epsilon_decay)
            steps_per_episode.append(steps)

        return steps_per_episode

    def plot_learning_progress(self, steps_per_episode, optimal_steps, window=20):
        """Plot learning curve with a moving average."""
        smoothed = np.convolve(steps_per_episode, np.ones(window)/window, mode='valid')
        plt.plot(smoothed, label=f'Moving avg ({window} episodes)')
        plt.axhline(y=optimal_steps, color='r', linestyle='--', label='Optimal')
        plt.title('DQN - Learning Progress')
        plt.xlabel('Episode')
        plt.ylabel('Steps')
        plt.legend()
        plt.show()


# === Main ===

env = GridWorldEnvironment()
agent = DQNAgent(env, hidden_size=32, eta=0.01, gamma=0.95,
                 buffer_size=3000, batch_size=32, target_update_freq=200)

# The optimal path is 8 steps (navigate around the wall via the gap at row 2).
optimal_steps = 8

# TODO: Uncomment after completing the implementation
# steps_history = agent.train(episodes=500, max_steps=100)
# agent.plot_learning_progress(steps_history, optimal_steps)


<details>
<summary>Show Solution</summary>

```python
import numpy as np
import matplotlib.pyplot as plt


def relu(x):
    return np.maximum(0, x)

def relu_derivative(x):
    return (x > 0).astype(float)


class GridWorldEnvironment:
    def __init__(self):
        self.grid_size = 5
        self.num_states = 25
        self.actions = 4
        self.start = (0, 0)
        self.goal = (4, 4)
        self.moves = {0: (-1, 0), 1: (1, 0), 2: (0, -1), 3: (0, 1)}
        self.walls = set()
        for r in range(self.grid_size):
            if r != 2:
                self.walls.add((r, 2))

    def reset(self):
        self.state = self.start
        return self.state[0] * self.grid_size + self.state[1]

    def step(self, action):
        dr, dc = self.moves[action]
        next_pos = (self.state[0] + dr, self.state[1] + dc)
        if (0 <= next_pos[0] < self.grid_size and
            0 <= next_pos[1] < self.grid_size and
            next_pos not in self.walls):
            self.state = next_pos
        idx = self.state[0] * self.grid_size + self.state[1]
        if self.state == self.goal:
            return idx, 10.0, True
        return idx, -1.0, False


class DQNAgent:
    def __init__(self, env, hidden_size=32, eta=0.01, gamma=0.95,
                 epsilon=1.0, epsilon_min=0.01, epsilon_decay=0.99,
                 buffer_size=3000, batch_size=32, target_update_freq=200):
        self.env = env
        self.hidden_size = hidden_size
        self.eta = eta
        self.gamma = gamma
        self.epsilon = epsilon
        self.epsilon_min = epsilon_min
        self.epsilon_decay = epsilon_decay
        self.batch_size = batch_size
        self.target_update_freq = target_update_freq
        self.buffer = ReplayBuffer(buffer_size)
        self._reset_weights()

    def _reset_weights(self):
        n_in = self.env.num_states
        n_h = self.hidden_size
        n_out = self.env.actions
        self.W1 = np.random.randn(n_in, n_h) * np.sqrt(2.0 / n_in)
        self.b1 = np.zeros(n_h)
        self.W2 = np.random.randn(n_h, n_out) * np.sqrt(2.0 / n_h)
        self.b2 = np.zeros(n_out)
        self._sync_target_network()

    def _sync_target_network(self):
        self.W1_target = self.W1.copy()
        self.b1_target = self.b1.copy()
        self.W2_target = self.W2.copy()
        self.b2_target = self.b2.copy()

    def _one_hot_batch(self, states):
        X = np.zeros((len(states), self.env.num_states))
        X[np.arange(len(states)), states] = 1
        return X

    def _forward(self, X, use_target=False):
        W1 = self.W1_target if use_target else self.W1
        b1 = self.b1_target if use_target else self.b1
        W2 = self.W2_target if use_target else self.W2
        b2 = self.b2_target if use_target else self.b2

        h1 = X @ W1 + b1
        hidden = relu(h1)
        q = hidden @ W2 + b2  # linear output for Q-values
        return q, {'X': X, 'h1': h1, 'hidden': hidden}

    def _backward(self, cache, targets):
        X = cache['X']
        h1 = cache['h1']
        hidden = cache['hidden']

        q, _ = self._forward(X)
        delta_output = targets - q
        delta_hidden = (delta_output @ self.W2.T) * relu_derivative(h1)

        self.W2 += self.eta * (hidden.T @ delta_output) / self.batch_size
        self.b2 += self.eta * np.mean(delta_output, axis=0)
        self.W1 += self.eta * (X.T @ delta_hidden) / self.batch_size
        self.b1 += self.eta * np.mean(delta_hidden, axis=0)

    # The epsilon-greedy logic lives directly in this method.
    # Notebook 04 shows an alternative: passing a separate policy
    # object into the agent, so you can swap policies without
    # changing the agent class.
    def select_action(self, state):
        if np.random.rand() < self.epsilon:
            return np.random.randint(self.env.actions)
        X = self._one_hot_batch(np.array([state]))
        q, _ = self._forward(X)
        return np.argmax(q[0])

    def update(self):
        if len(self.buffer) < self.batch_size:
            return

        states, actions, rewards, next_states, dones = self.buffer.sample(self.batch_size)

        X = self._one_hot_batch(states)
        X_next = self._one_hot_batch(next_states)

        q_values, cache = self._forward(X)
        next_q_values, _ = self._forward(X_next, use_target=True)

        targets = q_values.copy()
        td_targets = rewards + (1 - dones) * self.gamma * np.max(next_q_values, axis=1)
        targets[np.arange(self.batch_size), actions] = td_targets

        self._backward(cache, targets)

    def train(self, episodes=500, max_steps=100):
        steps_per_episode = []
        total_steps = 0
        for episode in range(episodes):
            state = self.env.reset()
            done = False
            steps = 0
            while not done and steps < max_steps:
                action = self.select_action(state)
                next_state, reward, done = self.env.step(action)
                self.buffer.store(state, action, reward, next_state, done)
                self.update()
                state = next_state
                steps += 1
                total_steps += 1
                if total_steps % self.target_update_freq == 0:
                    self._sync_target_network()
            self.epsilon = max(self.epsilon_min, self.epsilon * self.epsilon_decay)
            steps_per_episode.append(steps)
        return steps_per_episode

    def plot_learning_progress(self, steps_per_episode, optimal_steps, window=20):
        smoothed = np.convolve(steps_per_episode, np.ones(window)/window, mode='valid')
        plt.plot(smoothed, label=f'Moving avg ({window} episodes)')
        plt.axhline(y=optimal_steps, color='r', linestyle='--', label='Optimal')
        plt.title('DQN - Learning Progress')
        plt.xlabel('Episode')
        plt.ylabel('Steps')
        plt.legend()
        plt.show()


env = GridWorldEnvironment()
agent = DQNAgent(env, hidden_size=32, eta=0.01, gamma=0.95,
                 buffer_size=3000, batch_size=32, target_update_freq=200)
optimal_steps = 8

steps_history = agent.train(episodes=500, max_steps=100)
agent.plot_learning_progress(steps_history, optimal_steps)
```

</details>
